In [1]:
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

/home/cnouri/miniconda3/envs/fb-pa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
# Load your annotated dataframe
df_anno = pd.read_csv('../data/clean_annotated_comments.csv', low_memory=False)

# Inspect
print(df_anno.head())

            url_id                                          clean_url  \
0  4a5jagld15qh9m4  https://www.lenouveaudetective.com/dompierre-s...   
1  4a5jagld15qh9m4  https://www.lenouveaudetective.com/dompierre-s...   
2  4a5jagld15qh9m4  https://www.lenouveaudetective.com/dompierre-s...   
3  9kikqscza41dy1l  http://secretnews.fr/2018/01/25/ferrero-rappel...   
4  9kikqscza41dy1l  http://secretnews.fr/2018/01/25/ferrero-rappel...   

            parent_domain       source_type        theme  \
0  lenouveaudetective.com  sensationnaliste  fait divers   
1  lenouveaudetective.com  sensationnaliste  fait divers   
2  lenouveaudetective.com  sensationnaliste  fait divers   
3           secretnews.fr         parodique        santé   
4           secretnews.fr         parodique        santé   

   false_news_usr_feedback  hate_speech_usr_feedback  \
0                     79.0                      24.0   
1                     79.0                      24.0   
2                     79.0      

In [24]:
# Load your annotated dataframe
df_com = pd.read_csv('../data/clean_flag_comments.csv', low_memory=False)

# Inspect
print(df_com.head())

            url_id                                          clean_url  \
0  wsncss9p54zh5dh  http://silencescomplices.blogspot.com/2016/04/...   
1  wsncss9p54zh5dh  http://silencescomplices.blogspot.com/2016/04/...   
2  wsncss9p54zh5dh  http://silencescomplices.blogspot.com/2016/04/...   
3  wsncss9p54zh5dh  http://silencescomplices.blogspot.com/2016/04/...   
4  wsncss9p54zh5dh  http://silencescomplices.blogspot.com/2016/04/...   

                    parent_domain     theme  false_news_usr_feedback  \
0  silencescomplices.blogspot.com  religion                       65   
1  silencescomplices.blogspot.com  religion                       65   
2  silencescomplices.blogspot.com  religion                       65   
3  silencescomplices.blogspot.com  religion                       65   
4  silencescomplices.blogspot.com  religion                       65   

   hate_speech_usr_feedback      flag_type                       account_name  \
0                       178  False-et-Hate  Ami

In [25]:
# Load your annotated dataframe
df_post = pd.read_csv('../data/clean_flag_post.csv', low_memory=False)

# Inspect
print(df_post.head())

                        ct_id                                id  platform  \
0    9042110|1302115643288425  637517436414919_1302115643288425  Facebook   
1    8510743|1061589347344953  898813010289255_1061589347344953  Facebook   
2   10129610|2117554938361532  849855428464829_2117554938361532  Facebook   
3  12339664|10161950464175486     99815155485_10161950464175486  Facebook   
4    9036657|1028223214039890  428956983966519_1028223214039890  Facebook   

   type                                              title  \
0  link  La femme non voilée n'a pas d'honneur et mérit...   
1  link  La femme non voilée n'a pas d'honneur et mérit...   
2  link  La femme non voilée n'a pas d'honneur et mérit...   
3  link  La femme non voilée n'a pas d'honneur et mérit...   
4  link  La femme non voilée n'a pas d'honneur et mérit...   

                          caption  \
0  silencescomplices.blogspot.com   
1  silencescomplices.blogspot.com   
2  silencescomplices.blogspot.com   
3  silencescompl

In [26]:
print("Columns in df_anno:", df_anno.columns.tolist())
print("Columns in df_com:", df_com.columns.tolist())
print("Columns in df_post:", df_post.columns.tolist())

Columns in df_anno: ['url_id', 'clean_url', 'parent_domain', 'source_type', 'theme', 'false_news_usr_feedback', 'hate_speech_usr_feedback', 'account_name', 'moderation_charter', 'account_subscriber_count', 'page_group_type', 'post_url', 'share_title', 'post_id', 'comment_id', 'user_label', 'text', 'stop', 'formatted_date', 'reactions', 'replies', 'in_reply_to']
Columns in df_com: ['url_id', 'clean_url', 'parent_domain', 'theme', 'false_news_usr_feedback', 'hate_speech_usr_feedback', 'flag_type', 'account_name', 'account_subscriber_count', 'post_url', 'post_id', 'id', 'user_id', 'user_handle', 'user_url', 'user_label', 'text', 'html', 'formatted_date', 'date', 'reactions', 'replies', 'in_reply_to', 'page_group_type', 'moderation_charter', 'source_type']
Columns in df_post: ['ct_id', 'id', 'platform', 'type', 'title', 'caption', 'message', 'description', 'date', 'datetime', 'updated', 'link', 'post_url', 'score', 'video_length_ms', 'live_video_status', 'actual_like_count', 'expected_like

In [27]:
num_nans = df_anno['comment_id'].isna().sum()
print("Number of NaN values in df_anno['comment_id']:", num_nans)
num_nans = df_anno['post_id'].isna().sum()
print("Number of NaN values in df_anno['post_id']:", num_nans)
num_nans = df_anno['in_reply_to'].isna().sum()
print("Number of NaN values in df_anno['in_reply_to']:", num_nans)
num_nans = df_com['id'].isna().sum()
print("Number of NaN values in df_com['id']:", num_nans)
num_nans = df_post['id'].isna().sum()
print("Number of NaN values in df_post['id']:", num_nans)

Number of NaN values in df_anno['comment_id']: 12752
Number of NaN values in df_anno['post_id']: 77
Number of NaN values in df_anno['in_reply_to']: 28457
Number of NaN values in df_com['id']: 0
Number of NaN values in df_post['id']: 0


In [28]:
df_anno['post_id']

0        2403538653036604
1        2403538653036604
2        2403538653036604
3        2078575259054886
4        2078575259054886
               ...       
43300                 NaN
43301                 NaN
43302                 NaN
43303                 NaN
43304                 NaN
Name: post_id, Length: 43305, dtype: object

In [29]:
df_post['id']

0         637517436414919_1302115643288425
1         898813010289255_1061589347344953
2         849855428464829_2117554938361532
3            99815155485_10161950464175486
4         428956983966519_1028223214039890
                       ...                
30154      277506326438568_576893833166481
30155     357133244305148_2686324364719346
30156     910463402404598_2311158719001719
30157    1185383891642590_1238735602974085
30158     284129444969978_1744623452253896
Name: id, Length: 30159, dtype: object

In [21]:
# Step 1: Split the post IDs into two columns
df_post['id_first'] = df_post['id'].str.split('_').str[0]
df_post['id_second'] = df_post['id'].str.split('_').str[1]

# Step 2: Select and rename the columns to have the 'post_' prefix
df_post_renamed = df_post.rename(columns={
    'title': 'post_title',
    'description': 'post_description',
    'message': 'post_message'
})

# Step 3: Melt to long format using renamed columns
df_post_long = pd.melt(
    df_post_renamed,
    id_vars=['post_title', 'post_description', 'post_message'],
    value_vars=['id_first', 'id_second'],
    var_name='which_id',
    value_name='post_id'
)

# Step 4: Merge with df_anno on post_id
df_anno = df_anno.merge(
    df_post_long,
    on='post_id',
    how='left'
)

# Step 5: Print metrics
total = len(df_anno)
matched = df_anno['post_title'].notna().sum()
print(f"Total samples in df_anno: {total}")
print(f"Samples with matched post info: {matched}")
print(f"Match rate: {matched / total:.2%}")


Total samples in df_anno: 46338
Samples with matched post info: 34725
Match rate: 74.94%


,url_id,clean_url,parent_domain,source_type,theme,false_news_usr_feedback,hate_speech_usr_feedback,account_name,moderation_charter,account_subscriber_count,...,text,stop,formatted_date,reactions,replies,in_reply_to,title,description,message,which_id
0,4a5jagld15qh9m4,https://www.lenouveaudetective.com/dompierre-s...,lenouveaudetective.com,sensationnaliste,fait divers,79.0,24.0,AUX Portes DU Pouvoir,no_moderation,7962.0,...,"Pourquoi toutes ces disparitions de jeunes, c'...",no_stop,25 juin 2019,1,0,NaN,"Disparition d’Alexis, 16 ans : “Il faut que so...",NaN,NaN,id_second
1,4a5jagld15qh9m4,https://www.lenouveaudetective.com/dompierre-s...,lenouveaudetective.com,sensationnaliste,fait divers,79.0,24.0,AUX Portes DU Pouvoir,no_moderation,7962.0,...,"Pourquoi toutes ces disparitions de jeunes, c'...",no_stop,25 juin 2019,1,0,NaN,"Disparition d’Alexis, 16 ans : “Il faut que so...",NaN,NaN,id_second
2,4a5jagld15qh9m4,https://www.lenouveaudetective.com/dompierre-s...,lenouveaudetective.com,sensationnaliste,fait divers,79.0,24.0,AUX Portes DU Pouvoir,no_moderation,7962.0,...,ptg60,no_stop,25 juin 2019,0,0,NaN,"Disparition d’Alexis, 16 ans : “Il faut que so...",NaN,NaN,id_second
3,4a5jagld15qh9m4,https://www.lenouveaudetective.com/dompierre-s...,lenouveaudetective.com,sensationnaliste,fait divers,79.0,24.0,AUX Portes DU Pouvoir,no_moderation,7962.0,...,ptg60,no_stop,25 juin 2019,0,0,NaN,"Disparition d’Alexis, 16 ans : “Il faut que so...",NaN,NaN,id_second
4,4a5jagld15qh9m4,https://www.lenouveaudetective.com/dompierre-s...,lenouveaudetective.com,sensationnaliste,fait divers,79.0,24.0,AUX Portes DU Pouvoir,no_moderation,7962.0,...,Ptg dans le 13,no_stop,25 juin 2019,0,0,NaN,"Disparition d’Alexis, 16 ans : “Il faut que so...",NaN,NaN,id_second
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,pas grave sa!! 😂 😂,no_stop,NaN,0,0,171408106886677.0,Réforme orthographique : 2400 mots changent dè...,"Elle change, notre belle langue française… Et ...",NaN,id_second
46334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,pas grave sa!! 😂 😂,no_stop,NaN,0,0,171408106886677.0,Réforme orthographique : 2400 mots changent dè...,"Elle change, notre belle langue française… Et ...",Yolo l'orthographe ...,id_second
46335,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,pas grave sa!! 😂 😂,no_stop,NaN,0,0,171408106886677.0,Arabie saoudite : pour les femmes c'est pas gagné,L'histoire d'une Américaine coincée en Arabie ...,NaN,id_second
46336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,pas grave sa!! 😂 😂,no_stop,NaN,0,0,171408106886677.0,Un événement cosmique qui n'arrive que tous le...,Deux Lunes dans le ciel du 27 Juillet ! La pro...,NaN,id_second
